# Study Buddy

An intelligent study assistant that uses Retrieval-Augmented Generation (RAG) to answer questions from uploaded educational PDFs.

The project pipeline:

**PDF → Text Extraction → Cleaning → Chunking → Embeddings → FAISS → Retrieval → Qwen LLM → Grounded Answers → Kokoro TTS → Flashcards → Voice Chatbot**

---
Made By:
- Merna Mohamed
- Mohamed Mahmoud
- Ranya Farrag
- Youssef Abady

# 0. Environment Setup

This section installs **all required system and Python dependencies** in one place. Keeping installations at the beginning makes the rest of the notebook easier to run from top to bottom.

In [ ]:
# System dependency
!apt-get update -qq && apt-get install -y -qq espeak-ng

# Python dependencies
!pip install --no-deps \
    transformers accelerate bitsandbytes sentence-transformers pypdf faiss-cpu \
    spacy kokoro soundfile "misaki[en]" loguru num2words phonemizer dlinfo

# 1. RAG and LLM

This section builds the core Study Buddy pipeline. A PDF is converted into searchable chunks, the chunks are embedded and stored in FAISS, and relevant content is retrieved before Qwen generates a grounded answer.

In [ ]:
# Imports for the RAG and LLM section
import os
import re
import json
import numpy as np
import torch
import faiss

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
from google.colab import files


## 1.1 Model Configuration

Defines the local embedding and generation models used throughout the notebook. Qwen2.5-7B-Instruct is used for answer generation, while MiniLM creates compact semantic embeddings for retrieval.

In [ ]:
# Model configuration
# Swap to Qwen/Qwen2.5-3B-Instruct if the available GPU runs out of memory.

GENERATION_MODEL = "Qwen/Qwen2.5-7B-Instruct"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

print("Generation model:", GENERATION_MODEL)
print("Embedding model:", EMBEDDING_MODEL)


## 1.2 Load the Models

Loads the embedding model and Qwen language model. When CUDA is available, Qwen uses 4-bit quantization to reduce GPU memory usage.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading embedding model...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL, device="cpu")

print("Loading Qwen LLM...")
tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL)
llm_model = AutoModelForCausalLM.from_pretrained(
    GENERATION_MODEL,
    quantization_config=bnb_config if device == "cuda" else None,
    device_map="auto" if device == "cuda" else None,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)

print("Qwen LLM and embedding model loaded successfully!")


## 1.3 Qwen Text Generation

Provides a reusable function that sends a prompt to Qwen and returns the generated response.

In [ ]:
def generate_with_llm(prompt, max_new_tokens=1024, temperature=0.7):
    """Run a single-turn prompt through the local Qwen model."""
    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(llm_model.device)

    output_ids = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    response_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(response_ids, skip_special_tokens=True)

    return response.strip()


## 1.4 Upload the Study PDF

Uploads the educational PDF that Study Buddy will use as its source of knowledge.

In [ ]:
uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]

print(f"Uploaded PDF: {pdf_filename}")


## 1.5 Extract Text from the PDF

Reads the PDF page by page and keeps the extracted text together with its page number so retrieved answers can be traced back to the source.

In [ ]:
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    pages = []

    for page_number, page in enumerate(reader.pages):
        text = page.extract_text()

        if text:
            pages.append({
                "page": page_number + 1,
                "text": text
            })

    return pages


pages = extract_text_from_pdf(pdf_filename)

print("Number of pages:", len(pages))


## 1.6 Inspect Extracted Text

Displays a small sample of the extracted PDF text so the extraction result can be checked before preprocessing.

In [ ]:
for page in pages[:3]:
    print("=" * 80)
    print("PAGE:", page["page"])
    print(page["text"][:1500])


## 1.7 Clean the Text

Removes unnecessary line breaks and repeated whitespace. This produces cleaner text before the document is divided into chunks.

In [ ]:
def clean_text(text):
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


for page in pages:
    page["text"] = clean_text(page["text"])

print(pages[0]["text"][:1000])


## 1.8 Create Text Chunks

Splits each page into overlapping chunks. The overlap helps preserve context when an important idea falls near a chunk boundary.

In [ ]:
def create_chunks(pages, chunk_size=1000, overlap=200):
    chunks = []

    for page in pages:
        text = page["text"]
        start = 0

        while start < len(text):
            end = start + chunk_size
            chunk_text = text[start:end]

            chunks.append({
                "text": chunk_text,
                "page": page["page"]
            })

            start += chunk_size - overlap

    return chunks


chunks = create_chunks(pages)

print("Number of chunks:", len(chunks))


## 1.9 Inspect Chunks

Shows a few generated chunks and their source pages to verify that chunking worked as expected.

In [ ]:
for i, chunk in enumerate(chunks[:5]):
    print("=" * 80)
    print("CHUNK:", i)
    print("PAGE:", chunk["page"])
    print(chunk["text"])


## 1.10 Generate Chunk Embeddings

Converts every text chunk into a numerical vector using the Sentence-Transformer embedding model. These vectors allow semantic similarity searches.

In [ ]:
def create_embedding(text):
    embedding = embedding_model.encode(text, convert_to_numpy=True)
    return embedding.astype("float32")


embeddings = []

for i, chunk in enumerate(chunks):
    embedding = create_embedding(chunk["text"])
    embeddings.append(embedding)

    if (i + 1) % 10 == 0:
        print(f"Embedded {i + 1}/{len(chunks)} chunks")


## 1.11 Prepare and Normalize Embeddings

Converts the list of embeddings into a NumPy array and normalizes the vectors so inner-product similarity can be used as a cosine-similarity-style measure.

In [ ]:
embeddings = np.array(embeddings)

print("Embedding shape:", embeddings.shape)

faiss.normalize_L2(embeddings)

print("Embeddings normalized.")


## 1.12 Build the FAISS Vector Index

Creates a FAISS index and stores all normalized chunk embeddings in it. This becomes the searchable vector store for the RAG pipeline.

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("Vector database created!")
print("Number of vectors:", index.ntotal)


## 1.13 Semantic Retrieval

Embeds a student's question and searches FAISS for the most similar chunks from the PDF.

In [ ]:
def embed_query(query):
    embedding = embedding_model.encode(
        query,
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(embedding.reshape(1, -1))

    return embedding


def retrieve_relevant_chunks(query, top_k=5):
    query_embedding = embed_query(query)

    scores, indices = index.search(
        query_embedding.reshape(1, -1),
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "text": chunks[idx]["text"],
            "page": chunks[idx]["page"],
            "score": float(score)
        })

    return results


## 1.14 Test Retrieval

Runs a sample question and displays the highest-scoring PDF chunks returned by the vector search.

In [ ]:
question = "What is quick sort"

results = retrieve_relevant_chunks(question)

for result in results:
    print("=" * 80)
    print("PAGE:", result["page"])
    print("SIMILARITY:", round(result["score"], 3))
    print(result["text"])


## 1.15 Build the Retrieved Context

Combines the retrieved chunks into one context string, while keeping their page numbers. This context is passed to the language model.

In [ ]:
def build_context(results):
    context_parts = []

    for result in results:
        context_parts.append(
            f"[Page {result['page']}]\n"
            f"{result['text']}"
        )

    return "\n\n".join(context_parts)


## 1.16 Study Buddy System Prompt

Defines Study Buddy's teaching style and grounding rules. The model is instructed to answer from the retrieved PDF context rather than inventing outside information.

In [ ]:
SYSTEM_PROMPT = """
IDENTITY

You are StudyBuddy, a sharp and encouraging AI study coach. A student
has handed you a PDF they're trying to learn, and your job is to make
it click for them - not recite it back.

Every answer you give is read out loud by a text-to-speech voice and
also shown as plain text in a console. Because of that, two hard rules
override everything else in this prompt:
- Never use markdown symbols - no **, ##, -, *, `, > or bullet dashes.
- Never use emoji.
Write the way a real tutor talks out loud: plain sentences, natural
pauses from punctuation, nothing that only makes sense on a screen.

GROUNDING RULES

1. Answer using only the retrieved PDF context you're given. Do not add
   outside facts, even ones you're confident are true.
2. If the context doesn't contain the answer, say so plainly - for
   example: "I don't see that covered in this PDF." Never imply the
   PDF said something it didn't.
3. You may reference earlier turns in the conversation for continuity,
   but the PDF context is always the source of truth for facts.

HOW YOU TEACH

Pipeline: understand the question, simplify the idea, explain it
clearly, reinforce it so it sticks.

- Lead with plain language, then introduce the technical term once the
  idea already makes sense.
- Use a short real-world analogy only when it genuinely clarifies
  something - skip it if it would feel forced.
- Walk through processes as spoken sequence - "First... then...
  after that..." - never as a numbered or bulleted list.
- If the student seems confused or re-asks something, explain it from
  a different angle rather than repeating the same explanation.
- Keep answers tight: a few short spoken paragraphs by default. Only
  go longer if the student explicitly asks for more depth.

HOW YOU STRUCTURE EACH ANSWER

Open by answering the question directly in one or two sentences - no
restating the question, no warm-up.

Then unpack it: break the idea into its simplest parts, add an example
or analogy if it helps, and name any term worth remembering.

Close with the one thing you most want to stick, and only when it
genuinely fits, one short question to check understanding. Don't force
a question onto every answer.

VOICE

You sound like the best TA in the department: confident, warm, and
easy to talk to - not stiff, not childish, not trying too hard to be
funny. Encouraging without hype, precise without being cold.

This is an ongoing conversation, not a series of one-off questions.
Only open with a greeting if the conversation history is empty - if
the student has already asked something before this, skip straight to
answering. Never re-introduce yourself mid-session.

Use these only when they genuinely fit, and never the same one twice
in a row:

Opening a brand-new session (empty history only) - pick one:
"Alright, let's get into it - what are we studying today?"
"I've got the material loaded up. Where do you want to start?"
"Ready when you are - ask me anything from the PDF."

Marking real progress, sparingly, not every turn - pick one:
"Exactly - you're connecting the dots now."
"That's the right instinct."
"Good catch - that's the part most people miss."

Never say "as an AI" or reference being a language model. Never claim
the PDF said something you haven't actually seen in the context.
"""


## 1.17 Generate a Grounded Answer

Combines the system prompt and retrieved PDF context into the final prompt sent to Qwen.

In [ ]:
def generate_answer(question, results):
    context = build_context(results)

    prompt = f"""
{SYSTEM_PROMPT}

========================
RETRIEVED PDF CONTEXT
========================

{context}

========================
STUDENT QUESTION
========================

{question}

========================
INSTRUCTIONS
========================

Answer the student's question using the retrieved PDF context.
If the answer is not supported by the context, say so clearly.
"""

    return generate_with_llm(prompt)


## 1.18 Single-Question Test

Tests the complete retrieval → context → Qwen generation path for one question.

In [ ]:
question = input("🎓 Ask StudyBuddy: ")

results = retrieve_relevant_chunks(question, top_k=5)
answer = generate_answer(question, results)

print("\n" + "=" * 80)
print("🤖 STUDYBUDDY")
print("=" * 80)
print(answer)


## 1.19 Conversational Study Buddy

Adds conversation history to the RAG prompt so Study Buddy can understand follow-up questions while still grounding factual answers in the PDF.

In [ ]:
conversation_history = []


def study_buddy(question, top_k=5):
    results = retrieve_relevant_chunks(question, top_k=top_k)
    context = build_context(results)

    history = ""

    for message in conversation_history:
        history += f"""
Student: {message['question']}
StudyBuddy: {message['answer']}
"""

    prompt = f"""
{SYSTEM_PROMPT}

PREVIOUS CONVERSATION:

{history}

RETRIEVED PDF CONTEXT:

{context}

CURRENT STUDENT QUESTION:

{question}

Answer the student naturally while staying grounded in the PDF.
"""

    answer = generate_with_llm(prompt)

    conversation_history.append({
        "question": question,
        "answer": answer
    })

    return answer


## 1.20 Text-Only Study Loop

Provides a simple command-line study session where the student can keep asking questions until entering `exit`.

In [ ]:
print("🎓 StudyBuddy is ready!")
print("Type 'exit' to stop.\n")

while True:
    question = input("You: ")

    if question.lower() == "exit":
        print("Study session ended. Good luck! 🫡📚")
        break

    answer = study_buddy(question)

    print("\nStudyBuddy 🤖:", answer)
    print("\n" + "-" * 80 + "\n")


# 2. Text-to-Speech

This section adds Kokoro TTS so Study Buddy's generated answers can be spoken aloud. The text is cleaned first, then synthesized while preserving punctuation-based pauses.

In [ ]:
import re
import numpy as np
import soundfile as sf

from kokoro import KPipeline
from IPython.display import Audio, display

## 2.1 Voice Configuration and Model Loading

Selects the Kokoro voice and speed, then loads the TTS pipeline once so it can be reused for multiple answers.

In [ ]:
# Available voices include:
# af_heart, af_bella, af_nicole, af_sarah
# am_adam, am_michael
# bf_emma, bf_isabella
# bm_george, bm_lewis

TTS_VOICE = "af_heart"
TTS_SPEED = 1.05

tts = KPipeline(lang_code="a")

print("Kokoro TTS model loaded successfully!")


## 2.2 Clean Text for Speech

Removes markdown symbols, emojis, and excessive whitespace while keeping normal punctuation so the TTS model can produce natural pauses.

In [ ]:
def clean_for_speech(text):
    text = re.sub(r"[*_#>`]", "", text)
    text = re.sub(r"[\U0001F300-\U0001FAFF\u2600-\u27BF]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


## 2.3 Generate and Play Speech

Generates audio from the answer in a single Kokoro pipeline call, saves it as a WAV file, and provides notebook playback.

In [ ]:
def generate_speech(text, filename="output.wav"):
    text = clean_for_speech(text)

    if not text:
        return np.array([], dtype=np.float32), 24000

    audio_parts = []

    for _, _, audio in tts(
        text,
        voice=TTS_VOICE,
        speed=TTS_SPEED,
        split_pattern=r"(?<=[.!?])\s+"
    ):
        audio_parts.append(np.asarray(audio, dtype=np.float32))

    audio = np.concatenate(audio_parts)
    sampling_rate = 24000

    sf.write(filename, audio, sampling_rate)

    return audio, sampling_rate


def speak(text, filename="output.wav", autoplay=True):
    audio, sr = generate_speech(text, filename)

    print(f"Voice generated successfully: {filename}")
    display(Audio(audio, rate=sr, autoplay=autoplay))


## 2.4 TTS Quick Test

Confirms that Kokoro is loaded and can successfully convert a short Study Buddy response into speech.

In [ ]:
speak("Hey! I am StudyBuddy, and I am ready to help you study.")


## 2.5 Voice-Enabled StudyBuddy Loop

Combines the existing conversational RAG pipeline with Kokoro so every generated answer is both displayed and spoken aloud.

In [ ]:
def ask_study_buddy(question, top_k=5, speak_answer=True, filename="output.wav"):
    """Combine retrieval + Qwen generation + optional Kokoro speech."""

    answer = study_buddy(question, top_k=top_k)

    print("\n" + "=" * 80)
    print("🤖 STUDYBUDDY")
    print("=" * 80)
    print(answer)

    if speak_answer:
        speak(answer, filename=filename)

    return answer

In [ ]:
print("🎓 StudyBuddy voice chatbot is ready!")
print("Ask a question and it will answer out loud. Type 'exit' to stop.\n")

while True:
    question = input("You: ")

    if question.lower() == "exit":
        print("Study session ended. Good luck! 🫡📚")
        break

    ask_study_buddy(question)

    print("\n" + "-" * 80 + "\n")

# 3. Flashcard Generation

The Streamlit app samples chunks across the document, asks Qwen to generate a fixed number of question/answer cards in JSON, then parses and validates the model response. The notebook version uses the notebook's existing `chunks`, `tokenizer`, `llm_model`, and `generate_with_llm()` variables instead of Streamlit session state.

In [ ]:
# Imports for the Flashcard Generation section
import json
import re


## 3.1 Flashcard Prompt

Instructs Qwen to create concise flashcards using only the supplied PDF material and return them as a JSON array.

In [ ]:
FLASHCARD_PROMPT_TEMPLATE = """You are helping a student build study flashcards from their course material.

Using ONLY the material below, generate {num_cards} flashcards that cover the
most important concepts, definitions, and facts. Each flashcard must have a
short, clear "question" (or term) and a concise "answer".

MATERIAL:
{context}

Respond with ONLY a valid JSON array, no other text, no markdown code fences.
Format:
[
  {{"question": "...", "answer": "..."}},
  {{"question": "...", "answer": "..."}}
]
"""

## 3.2 Generate Flashcards

Samples chunks across the document rather than only taking the beginning, sends them to Qwen, extracts the JSON array from the response, and keeps only cards containing both a question and an answer.

This is the notebook-adapted version of the `generate_flashcards()` function from `app.py`.

In [ ]:
def generate_flashcards(num_cards=8, sample_chunks=12):
    # Spread the sample across the document instead of using only the first chunks.
    step = max(1, len(chunks) // sample_chunks)
    sample = chunks[::step][:sample_chunks]

    context = "\n\n".join(c["text"] for c in sample)

    prompt = FLASHCARD_PROMPT_TEMPLATE.format(
        num_cards=num_cards,
        context=context
    )

    raw = generate_with_llm(
        prompt,
        max_new_tokens=1500,
        temperature=0.5
    )

    # Remove markdown code fences if the model adds them.
    raw = re.sub(
        r"^```(json)?|```$",
        "",
        raw.strip(),
        flags=re.MULTILINE
    ).strip()

    # Extract the JSON array if the model included extra text.
    match = re.search(r"\[.*\]", raw, flags=re.DOTALL)

    if match:
        raw = match.group(0)

    try:
        cards = json.loads(raw)
        cards = [
            card for card in cards
            if "question" in card and "answer" in card
        ]
    except json.JSONDecodeError:
        cards = []

    return cards


## 3.3 Generate and Inspect Flashcards

Runs the extracted flashcard-generation logic and displays the resulting cards. Adjust `num_cards` to control how many cards are requested.

In [ ]:
num_cards = 8

flashcards = generate_flashcards(num_cards=num_cards)

if not flashcards:
    print("Couldn't parse flashcards from the model's response. Try again.")
else:
    for i, card in enumerate(flashcards, start=1):
        print("=" * 80)
        print(f"CARD {i}")
        print("QUESTION:", card["question"])
        print("ANSWER:", card["answer"])


# 4. Quiz Generation

In [ ]:
QUIZ_MODES = {
    "EZ": """
Generate easy questions.
Focus on:
- definitions
- direct facts
- basic concepts
- simple understanding

Questions should be straightforward and answerable directly from the provided material.
""",

    "Tuff": """
Generate medium-difficulty questions.
Focus on:
- understanding concepts
- comparisons
- relationships between ideas
- simple applications
- moderate reasoning

Avoid questions that are simply copied word-for-word from the material.
""",

    "Charlie Kirk": """
Generate very difficult questions.
Focus on:
- deep understanding
- subtle distinctions
- applying concepts
- multi-step reasoning
- distinguishing between closely related ideas
- plausible but incorrect distractors

Do NOT make questions difficult by using obscure information.
Make them difficult because they require genuine understanding of the material.
"""
}

In [ ]:
import json
import re

def generate_quiz(context, difficulty="EZ", num_questions=10):

    prompt = f"""
You are a quiz generator for an AI study assistant.

The quiz MUST be based ONLY on the provided study material.

Difficulty:
{QUIZ_MODES[difficulty]}

Generate exactly {num_questions} multiple-choice questions.

Each question must have:
- One question
- Exactly 4 options
- Exactly ONE correct answer
- A short explanation

Return ONLY valid JSON in this format:

{{
    "questions": [
        {{
            "question": "Question here",
            "options": [
                "Option A",
                "Option B",
                "Option C",
                "Option D"
            ],
            "correct_answer": 0,
            "explanation": "Why this answer is correct."
        }}
    ]
}}

IMPORTANT:
- correct_answer must be 0, 1, 2, or 3.
- Do not use information outside the study material.
- Do not create trick questions unless the difficulty requires it.
- Every question must have one clearly correct answer.

STUDY MATERIAL:
{context}
"""

    raw = generate_with_llm(
        prompt,
        max_new_tokens=2000,
        temperature=0.6
    )

    # Remove markdown code fences if the model adds them.
    raw = re.sub(
        r"^```(json)?|```$",
        "",
        raw.strip(),
        flags=re.MULTILINE
    ).strip()

    # Extract the JSON object if the model included extra text.
    match = re.search(r"\{.*\}", raw, flags=re.DOTALL)

    if match:
        raw = match.group(0)

    try:
        quiz = json.loads(raw)
        questions = quiz.get("questions", [])

        # Keep only well-formed questions.
        valid_questions = []
        for q in questions:
            if (
                "question" in q
                and "options" in q
                and len(q["options"]) == 4
                and "correct_answer" in q
                and q["correct_answer"] in [0, 1, 2, 3]
            ):
                valid_questions.append(q)

        questions = valid_questions

    except json.JSONDecodeError:
        questions = []

    return questions

In [ ]:
difficulty = "EZ"   # "EZ", "Tuff", or "Charlie Kirk"
num_questions = 20

quiz_context = "\n\n".join(c["text"] for c in chunks[:12])

quiz_questions = generate_quiz(
    context=quiz_context,
    difficulty=difficulty,
    num_questions=num_questions
)

if not quiz_questions:
    print("Couldn't parse a quiz from the model's response. Try again.")
else:
    for i, q in enumerate(quiz_questions, start=1):
        print("=" * 80)
        print(f"Q{i}: {q['question']}")
        for j, opt in enumerate(q["options"]):
            print(f"   {j}. {opt}")
        print("Correct:", q["correct_answer"], "-", q["explanation"])

In [ ]:
def take_quiz(questions):
    score = 0
    answered = 0

    for i, q in enumerate(questions, start=1):
        print("=" * 80)
        print(f"Q{i}: {q['question']}")
        for j, opt in enumerate(q["options"]):
            print(f"   {j}. {opt}")

        while True:
            answer = input("Your answer (0-3, or 'exit'): ").strip().lower()
            if answer == "exit":
                break
            if answer in ["0", "1", "2", "3"]:
                answer = int(answer)
                break
            print("Please enter 0, 1, 2, or 3 (or 'exit' to quit).")

        if answer == "exit":
            print("Quiz ended early.")
            break

        answered += 1
        if answer == q["correct_answer"]:
            print("Correct!")
            score += 1
        else:
            print(f"Incorrect. Correct answer: {q['correct_answer']} - {q['options'][q['correct_answer']]}")

        print("Explanation:", q["explanation"])

    print("=" * 80)
    print(f"Final score: {score}/{answered}")

    if answered > 0:
        if (score / answered) >= 0.5:
            print("You win!")
        else:
            print("You lose.")

take_quiz(quiz_questions)